# Student-selected fixed recency-weighted three-model blend

**12 October 2026.** After the student reported **0.94787** for the recency-weighted LightGBM 5/50 blend, the student explicitly selected this existing fixed recipe from Claude's supplied `original_workspace/run_recency.py`:

- **50% XGBoost**, 200 trees, 15 leaves, 127 bins, minimum child weight 1, L2 penalty 1.
- **25% LightGBM leaf-5**, 150 trees, 15 leaves, 63 bins, minimum child weight 0.001, L2 penalty 1.
- **25% LightGBM leaf-50**, otherwise identical to leaf-5.

All use learning rate 0.05, seed 42, two CPU threads and recency half-life **7**. Average row weight is normalised to one within each fitting period. XGBoost retains the six existing exclusions; both LightGBMs use all 165 features. No feature, tree-count, seed, half-life or blend-weight search occurs.

This keeps the XGBoost share of the **0.94989 public-best recent7 pair**, while replacing half of its LightGBM leaf-5 contribution with leaf-50. The three probabilities are averaged for every transaction; the AUC is then calculated from the new rankings. The formula is not an average of the component AUCs.

Freshly fit all three components on each of the same 15 checking windows: fit through k=7..21, then check k+1..k+14. The same three fitted components also produce fresh controls for the existing recent7 pair and the now-submitted LightGBM-only blend. The final fit uses all 31,235 confirmed labels through 35 and predicts every supplied test row. Unknown labels remain unknown.

**Validation limitation:** this experiment preserves the original supplied recipe, including its fixed six-feature exclusion selected using steps 1–32. That selection was not contained within every earlier fitting period. All fitting/checking rows are chronological, but feature selection in the inherited recipe is not fully nested. The windows overlap and have been reused for development. Their AUCs are not calibrated public-score predictions or proof of 0.96+. The final test fit uses no test rows or answers for fitting.

The 0.94787 result is the student's confirmed report for `submission_recent7_lgb_leaf5_50.csv`; the linked Kaggle page was inaccessible to the web tool, so it is recorded as a student report rather than an independently retrieved score. The screenshot-confirmed public best remains **0.94989** for `submission_lowreg_pair_recent7.csv`.

**Student choice and AI disclosure:** the student selected “Test the fixed three-model blend (Recommended)” after reviewing the candidate mixture. Claude supplied the existing recipe; OpenAI Codex implemented this standalone test, fitted only these fixed settings, checked artifacts and drafted the record. The student remains responsible for interpretation, final selection, submission and disclosure. No Kaggle upload or automatic next experiment is performed.

Run from the Kaggle project folder with the project Python environment. The fitting and prediction code below requires only the supplied train/test/sample files and model libraries. No old predictions or fitted models are needed. DATA_DIR and OUTPUT_DIR can be changed for a different environment.


In [1]:
from pathlib import Path
import os, json, hashlib, time, platform
import numpy as np
import pandas as pd
import xgboost, lightgbm, sklearn
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier, Booster
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr
from threadpoolctl import threadpool_limits

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / 'Data Set').is_dir():
    PROJECT_DIR = Path('/Users/mlvlo/BT4012/Kaggle')
DATA_DIR = PROJECT_DIR / 'Data Set'
OUTPUT_DIR = Path(os.environ.get('BT4012_THREE_MODEL_OUTPUT_DIR',
                                str(PROJECT_DIR / 'recency_three_model_results'))).resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FEATURES = [f'feat_{i}' for i in range(1, 166)]
TIME_EXCLUSIONS = ['feat_100','feat_101','feat_103','feat_136','feat_137','feat_139']
XGB_FEATURES = [c for c in FEATURES if c not in TIME_EXCLUSIONS]
CHECKS = [(k, k+1, k+14) for k in range(7, 22)]
SEED, THREADS, HALF_LIFE = 42, 2, 7
BLEND_WEIGHTS = {'xgboost':0.5, 'lightgbm_leaf5':0.25, 'lightgbm_leaf50':0.25}
FIXED_XGB = dict(n_estimators=200, learning_rate=0.05, tree_method='hist',
    grow_policy='lossguide', max_depth=0, max_leaves=15, max_bin=127,
    min_child_weight=1, reg_lambda=1, objective='binary:logistic',
    n_jobs=THREADS, random_state=SEED)
FIXED_LGB = dict(n_estimators=150, learning_rate=0.05, num_leaves=15, max_bin=63,
    min_child_weight=0.001, reg_lambda=1, objective='binary', deterministic=True,
    force_col_wise=True, verbosity=-1, n_jobs=THREADS, random_state=SEED)
STARTED = time.perf_counter()
def digest(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def progress(phase, **fields):
    fields.update(phase=phase, elapsed_seconds=round(time.perf_counter()-STARTED,3))
    (OUTPUT_DIR / 'progress.json').write_text(json.dumps(fields,indent=2)+'\n')
train_path = DATA_DIR / 'Train Data.csv'
test_path = DATA_DIR / 'Test Data.csv'
sample_path = DATA_DIR / 'Sample Submission/BT4012 Competition Submission.csv'
SOURCE_HASHES = {str(p):digest(p) for p in [train_path,test_path,sample_path]}
recipe = dict(half_life=HALF_LIFE, blend_weights=BLEND_WEIGHTS,
    fixed_xgboost=FIXED_XGB, fixed_lightgbm=FIXED_LGB, leaf_sizes=[5,50],
    xgboost_features=XGB_FEATURES, lightgbm_features=FEATURES, chronological_checks=CHECKS,
    source_hashes=SOURCE_HASHES, student_choice='Test the fixed three-model blend (Recommended)',
    public_best=0.94989, reported_weighted_lgb_public_auc=0.94787,
    tuning_search=False, kaggle_upload=False)
recipe_path = OUTPUT_DIR / 'declared_recipe.json'
if recipe_path.exists():
    assert json.loads(recipe_path.read_text()) == json.loads(json.dumps(recipe)), 'Settings or data changed.'
else:
    recipe_path.write_text(json.dumps(recipe,indent=2)+'\n')
print('Fixed chosen blend:', BLEND_WEIGHTS, '; recency half-life:', HALF_LIFE)

Fixed chosen blend: {'xgboost': 0.5, 'lightgbm_leaf5': 0.25, 'lightgbm_leaf50': 0.25} ; recency half-life: 7


In [2]:
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sample = pd.read_csv(sample_path)
labels = pd.to_numeric(train['label'], errors='coerce')
assert labels.dropna().isin([0,1]).all()
known = train.loc[labels.notna()].copy()
known['label'] = labels.loc[labels.notna()].astype(int)
assert len(known)==31235 and int(labels.isna().sum())==110537
assert known.time_step.between(1,35).all() and test.time_step.between(36,49).all()
assert known.time_step.max()<test.time_step.min()
assert train.txId.is_unique and test.txId.is_unique
assert set(train.txId).isdisjoint(set(test.txId))
assert test['index'].is_unique and sample['index'].is_unique and len(test)==len(sample)==15329
assert set(test['index'])==set(sample['index'])
assert np.isfinite(train[FEATURES].to_numpy()).all() and np.isfinite(test[FEATURES].to_numpy()).all()

def weights_for(learning):
    ages = learning.time_step.max() - learning.time_step
    weights = (0.5 ** (ages / HALF_LIFE)).to_numpy()
    weights = weights / weights.mean()
    assert np.isfinite(weights).all() and (weights>0).all()
    assert abs(weights.mean()-1)<1e-12
    return weights
def fit_components(learning):
    weights = weights_for(learning)
    xgb = XGBClassifier(**FIXED_XGB)
    leaf5 = LGBMClassifier(min_child_samples=5, **FIXED_LGB)
    leaf50 = LGBMClassifier(min_child_samples=50, **FIXED_LGB)
    with threadpool_limits(limits=THREADS):
        xgb.fit(learning[XGB_FEATURES], learning.label, sample_weight=weights)
        leaf5.fit(learning[FEATURES], learning.label, sample_weight=weights)
        leaf50.fit(learning[FEATURES], learning.label, sample_weight=weights)
    assert list(xgb.classes_)==list(leaf5.classes_)==list(leaf50.classes_)==[0,1]
    assert leaf5.n_estimators_==leaf50.n_estimators_==150
    return xgb, leaf5, leaf50
def component_predictions(models, rows):
    xgb, leaf5, leaf50 = models
    with threadpool_limits(limits=THREADS):
        px = xgb.predict_proba(rows[XGB_FEATURES])[:,1]
        p5 = leaf5.predict_proba(rows[FEATURES])[:,1]
        p50 = leaf50.predict_proba(rows[FEATURES])[:,1]
    return px, p5, p50
def combine(px, p5, p50):
    # Preserve the arithmetic grouping in Claude's original supplied recipe.
    leaf_blend = np.mean([p5,p50],axis=0)
    scores = {'recent7_pair':0.5*px+0.5*p5,
              'recent7_leaf_blend':leaf_blend,
              'recent7_three_model':0.5*px+0.5*leaf_blend}
    for p in scores.values():
        assert np.isfinite(p).all() and ((p>=0)&(p<=1)).all()
    assert np.allclose(scores['recent7_three_model'],0.5*px+0.25*p5+0.25*p50,rtol=0,atol=1e-15)
    return scores
print('Confirmed fitting labels:',len(known),'; unknown rows excluded:',int(labels.isna().sum()))

Confirmed fitting labels: 31235 ; unknown rows excluded: 110537


In [3]:
results_path = OUTPUT_DIR / 'validation_results.csv'
arrays_path = OUTPUT_DIR / 'validation_probabilities.npz'
records, arrays = [], {}
if results_path.exists():
    assert arrays_path.exists()
    records = pd.read_csv(results_path).to_dict('records')
    with np.load(arrays_path,allow_pickle=False) as old:
        arrays = {k:old[k].copy() for k in old.files}
    assert len(records)==len({int(row['cutoff']) for row in records})
for cutoff,first,last in CHECKS:
    fitting = known.loc[known.time_step<=cutoff]
    checking = known.loc[known.time_step.between(first,last)]
    assert fitting.time_step.max()<checking.time_step.min()
    assert set(fitting.txId).isdisjoint(checking.txId)
    assert fitting.label.nunique()==checking.label.nunique()==2
    if any(int(row['cutoff'])==cutoff for row in records):
        assert np.array_equal(arrays[f'labels_{cutoff}'],checking.label.to_numpy())
        assert np.array_equal(arrays[f'txIds_{cutoff}'],checking.txId.to_numpy())
        continue
    progress('fitting chosen fixed components',cutoff=cutoff,completed_checks=len(records),total_checks=15)
    models = fit_components(fitting)
    px,p5,p50 = component_predictions(models,checking)
    scores = combine(px,p5,p50)
    row = dict(cutoff=cutoff, validation_start=first, validation_end=last,
               fitting_rows=len(fitting), checking_rows=len(checking),
               checking_illicit=int(checking.label.sum()))
    for name,p in scores.items():
        row[name+'_auc'] = float(roc_auc_score(checking.label,p))
        arrays[f'{name}_{cutoff}'] = p
    for name,p in [('xgboost',px),('lightgbm_leaf5',p5),('lightgbm_leaf50',p50)]:
        arrays[f'{name}_{cutoff}'] = p
    arrays[f'labels_{cutoff}'] = checking.label.to_numpy()
    arrays[f'txIds_{cutoff}'] = checking.txId.to_numpy()
    arrays[f'time_steps_{cutoff}'] = checking.time_step.to_numpy()
    records.append(row)
    np.savez_compressed(arrays_path,**arrays)
    pd.DataFrame(records).to_csv(results_path,index=False)
    print(f"Cutoff {cutoff}: recent7 pair {row['recent7_pair_auc']:.6f}; "
          f"LightGBM blend {row['recent7_leaf_blend_auc']:.6f}; "
          f"fixed three-model {row['recent7_three_model_auc']:.6f}",flush=True)
validation = pd.DataFrame(records).sort_values('cutoff').reset_index(drop=True)
assert validation.cutoff.tolist()==list(range(7,22))
progress('chronological checking complete',completed_checks=15,total_checks=15)
display(validation[['cutoff','recent7_pair_auc','recent7_leaf_blend_auc','recent7_three_model_auc']])

Cutoff 7: recent7 pair 0.993837; LightGBM blend 0.992928; fixed three-model 0.993814
Cutoff 8: recent7 pair 0.994389; LightGBM blend 0.993828; fixed three-model 0.994382
Cutoff 9: recent7 pair 0.992803; LightGBM blend 0.992652; fixed three-model 0.992979
Cutoff 10: recent7 pair 0.976693; LightGBM blend 0.977536; fixed three-model 0.976995
Cutoff 11: recent7 pair 0.974708; LightGBM blend 0.976843; fixed three-model 0.976023
Cutoff 12: recent7 pair 0.973664; LightGBM blend 0.978060; fixed three-model 0.977228
Cutoff 13: recent7 pair 0.976168; LightGBM blend 0.977212; fixed three-model 0.977853
Cutoff 14: recent7 pair 0.974018; LightGBM blend 0.974775; fixed three-model 0.974176
Cutoff 15: recent7 pair 0.938092; LightGBM blend 0.944504; fixed three-model 0.940084
Cutoff 16: recent7 pair 0.948454; LightGBM blend 0.953096; fixed three-model 0.950482
Cutoff 17: recent7 pair 0.944733; LightGBM blend 0.946929; fixed three-model 0.942294
Cutoff 18: recent7 pair 0.929304; LightGBM blend 0.931286

,cutoff,recent7_pair_auc,recent7_leaf_blend_auc,recent7_three_model_auc
0,7,0.993837,0.992928,0.993814
1,8,0.994389,0.993828,0.994382
2,9,0.992803,0.992652,0.992979
3,10,0.976693,0.977536,0.976995
4,11,0.974708,0.976843,0.976023
5,12,0.973664,0.978060,0.977228
6,13,0.976168,0.977212,0.977853
7,14,0.974018,0.974775,0.974176
8,15,0.938092,0.944504,0.940084
9,16,0.948454,0.953096,0.950482


In [4]:
GROUPS = {'all_15':list(range(7,22)),'earlier_eight':list(range(7,15)),
          'later_seven':list(range(15,22)),'selected_four':[14,17,19,21],'latest':[21]}
VARIANTS = ('recent7_pair','recent7_leaf_blend','recent7_three_model')
summaries = []
for group,cutoffs in GROUPS.items():
    rows = validation.loc[validation.cutoff.isin(cutoffs)]
    for name in VARIANTS:
        values = rows[name+'_auc']
        record = dict(group=group,model=name,checks=len(rows),mean_auc=float(values.mean()),
                      worst_auc=float(values.min()))
        for suffix,reference in [('public_best','recent7_pair'),('weighted_lgb','recent7_leaf_blend')]:
            difference = values-rows[reference+'_auc']
            record['mean_delta_vs_'+suffix] = float(difference.mean())
            record['wins_vs_'+suffix] = int((difference>0).sum())
        summaries.append(record)
summary = pd.DataFrame(summaries)
summary.to_csv(OUTPUT_DIR / 'summary.csv',index=False)
display(summary.loc[summary.group.isin(['all_15','later_seven','selected_four','latest'])])

,group,model,checks,mean_auc,worst_auc,mean_delta_vs_public_best,wins_vs_public_best,mean_delta_vs_weighted_lgb,wins_vs_weighted_lgb
0,all_15,recent7_pair,15,0.955913,0.896894,0.000000,0,-0.002446,3
1,all_15,recent7_leaf_blend,15,0.958359,0.906131,0.002446,12,0.000000,0
2,all_15,recent7_three_model,15,0.956681,0.901247,0.000768,10,-0.001678,4
6,later_seven,recent7_pair,7,0.926059,0.896894,0.000000,0,-0.004162,0
7,later_seven,recent7_leaf_blend,7,0.930221,0.906131,0.004162,7,0.000000,0
8,later_seven,recent7_three_model,7,0.926680,0.901247,0.000621,4,-0.003541,0
9,selected_four,recent7_pair,4,0.935922,0.912455,0.000000,0,-0.001905,0
10,selected_four,recent7_leaf_blend,4,0.937827,0.912977,0.001905,4,0.000000,0
11,selected_four,recent7_three_model,4,0.935143,0.910630,-0.000779,2,-0.002684,0
12,latest,recent7_pair,1,0.912482,0.912482,0.000000,0,-0.004145,0


In [5]:
progress('full confirmed-label fit',known_rows=len(known))
final_models = fit_components(known)
px,p5,p50 = component_predictions(final_models,test)
full = combine(px,p5,p50)
arrays_to_save = {'indices':test['index'].to_numpy(),'txIds':test.txId.to_numpy(),
                 'xgboost':px,'lightgbm_leaf5':p5,'lightgbm_leaf50':p50,**full}
np.savez_compressed(OUTPUT_DIR / 'test_probabilities.npz',**arrays_to_save)
probabilities = pd.Series(full['recent7_three_model'],index=test['index'])
submission = sample[['index']].copy()
submission['target'] = submission['index'].map(probabilities)
assert submission.columns.tolist()==['index','target']
assert submission['index'].equals(sample['index']) and len(submission)==15329
assert np.isfinite(submission.target).all() and submission.target.between(0,1).all()
csv_path = OUTPUT_DIR / 'submission_recent7_xgb_lgb_leaf5_50_experimental.csv'
submission.to_csv(csv_path,index=False)
reread = pd.read_csv(csv_path)
assert reread['index'].equals(sample['index'])
assert np.allclose(reread.target,submission.target,rtol=0,atol=1e-15)

xgb,leaf5,leaf50 = final_models
xgb_path = OUTPUT_DIR / 'xgboost_recent7.json'
leaf5_path = OUTPUT_DIR / 'lightgbm_leaf5_recent7.txt'
leaf50_path = OUTPUT_DIR / 'lightgbm_leaf50_recent7.txt'
xgb.save_model(xgb_path)
leaf5.booster_.save_model(str(leaf5_path))
leaf50.booster_.save_model(str(leaf50_path))
loaded_xgb = XGBClassifier(n_jobs=THREADS)
loaded_xgb.load_model(xgb_path)
loaded_leaf5 = Booster(model_file=str(leaf5_path))
loaded_leaf50 = Booster(model_file=str(leaf50_path))
with threadpool_limits(limits=THREADS):
    rx = loaded_xgb.predict_proba(test[XGB_FEATURES])[:,1]
    r5 = loaded_leaf5.predict(test[FEATURES])
    r50 = loaded_leaf50.predict(test[FEATURES])
assert np.array_equal(rx,px)
assert np.allclose(r5,p5,rtol=0,atol=1e-15)
assert np.allclose(r50,p50,rtol=0,atol=1e-15)
assert np.allclose(combine(rx,r5,r50)['recent7_three_model'],full['recent7_three_model'],rtol=0,atol=1e-15)

public_records = [
    ('submission_lowreg_pair_recent7.csv',0.94989,'screenshot-confirmed'),
    ('submission_recent7_lgb_leaf5_50.csv',0.94787,'student-confirmed report'),
    ('submission_xgboost_lightgbm_lowreg.csv',0.94701,'screenshot-confirmed'),
    ('submission_xgboost_tuned.csv',0.94265,'screenshot-confirmed'),
    ('submission_hist_bin_ensemble.csv',0.94199,'screenshot-confirmed'),
    ('submission_hist_gradient_boosting.csv',0.93654,'screenshot-confirmed'),
    ('submission_extra_trees.csv',0.91673,'screenshot-confirmed'),
    ('submission_xgboost_lookalike_blend.csv',0.87805,'screenshot-confirmed'),
]
pd.DataFrame(public_records,columns=['submission_filename','public_auc','evidence_type']).to_csv(
    OUTPUT_DIR / 'reported_public_results.csv',index=False)
for p,expected in SOURCE_HASHES.items(): assert digest(p)==expected
metadata = dict(student_choice='Test the fixed three-model blend (Recommended)',
    half_life=HALF_LIFE,blend_weights=BLEND_WEIGHTS,fixed_xgboost=FIXED_XGB,
    fixed_lightgbm=FIXED_LGB,leaf_sizes=[5,50],xgboost_exclusions=TIME_EXCLUSIONS,
    chronological_checks=CHECKS,known_fitting_rows=len(known),unknown_rows_excluded=int(labels.isna().sum()),
    test_rows=len(test),source_hashes=SOURCE_HASHES,original_data_unchanged=True,
    validation_fits_fresh=True,prior_predictions_used_for_fitting=False,
    models_reload_verified=True,submission_sha256=digest(csv_path),
    saved_model_hashes={p.name:digest(p) for p in [xgb_path,leaf5_path,leaf50_path]},
    existing_public_best=0.94989,reported_weighted_lgb_public_auc=0.94787,new_candidate_public_auc=None,
    parameter_search=False,blend_weight_search=False,kaggle_upload=False,final_selection='pending student review',
    future_test_rows_used_for_fitting=False,external_data_used=False,unknown_rows_given_labels=False,
    validation_limitations=['Fixed inherited six-feature mask was chosen using steps 1-32; not nested within every earlier fitting window.',
                           'Overlapping checking periods have been reused for development; no independent held-out score forecast.'],
    versions=dict(python=platform.python_version(),numpy=np.__version__,pandas=pd.__version__,
                  sklearn=sklearn.__version__,xgboost=xgboost.__version__,lightgbm=lightgbm.__version__),
    elapsed_seconds=round(time.perf_counter()-STARTED,3))
(OUTPUT_DIR / 'experiment_metadata.json').write_text(json.dumps(metadata,indent=2)+'\n')
progress('complete',completed_checks=15,test_predictions=len(submission))
print('Saved:',csv_path)
print('All three reloaded models reproduce the final predictions. No Kaggle upload occurred.')

Saved: /Users/mlvlo/.codex/.chatgpt-projects/g-p-6a901a23216481918fe5444e40decc24/student_model_review/recency_three_model_results/submission_recent7_xgb_lgb_leaf5_50_experimental.csv
All three reloaded models reproduce the final predictions. No Kaggle upload occurred.


## Measured findings

**12 October 2026.** The student explicitly confirmed that **0.94787** belongs to submission_recent7_lgb_leaf5_50.csv. This is **0.00202 below** the screenshot-confirmed best of **0.94989** for the recent7 XGBoost/LightGBM pair, and **0.00086 above** the earlier unweighted pair's 0.94701. The supplied Kaggle submissions link was inaccessible to the web tool; the new observation is recorded as the student's confirmed report, not as independently retrieved page content.

The weighted LightGBM blend that Codex recommended did not improve the public best. Its local advantage over the recent7 pair did not transfer to this public subset. The comparison supports the existing XGBoost/LightGBM combination over this particular two-LightGBM recipe on the public subset under seven-step weighting. It does not isolate which component or feature difference caused the result, identify the optimal recency setting, establish a private result or rule out all LightGBM ensembles.

After this report, the student selected **“Test the fixed three-model blend (Recommended)”**: preserve XGBoost's 50% share, assign 25% each to the leaf-5 and leaf-50 LightGBMs, and retain half-life seven and all other supplied settings. This fixed recipe already appeared in Claude's original workspace source; no additional settings were searched.

All **15 existing chronological checks** and the full fit completed in **40.1 seconds** on CPU. All components and comparison controls were freshly fitted, with no previous probabilities used for fitting. The final fit used **31,235 confirmed labels** and generated **15,329 aligned test predictions**. **110,537 unknown rows** were excluded rather than assigned guessed labels.

| Check group | Submitted recent7 pair (0.94989 public) | Submitted weighted LightGBM blend (0.94787 public) | New fixed three-model blend |
|---|---:|---:|---:|
| All 15 | 0.955913 | 0.958359 | 0.956681 |
| Earlier eight | 0.982035 | 0.982979 | 0.982931 |
| Later seven | 0.926059 | 0.930221 | 0.926680 |
| Previously chosen four | 0.935922 | 0.937827 | 0.935143 |
| Latest: train 1–21, check 22–35 | 0.912482 | 0.916627 | 0.913471 |

Against the recent7 public-best pair, the new blend's all-15 mean changes by **+0.000768 AUC**, winning **10 of 15** checks. The later-seven mean changes by **+0.000621**, winning **4 of seven**. Its latest check improves **+0.000988**, but the previously selected-four mean falls **-0.000779**. Its worst check improves from **0.896894** to **0.901247**. These are small, mixed development gains, not uniform improvements.

**AI-assisted assessment:** The fixed three-model recipe has small, mixed development gains: higher overall, later-seven, latest and worst-check scores than the public-best pair, but a lower mean on the four originally chosen checks and only four later-check wins. It is an experimental candidate for student review, not a confidently superior Kaggle prediction. The preceding LightGBM-only blend had stronger local gains but scored lower publicly, demonstrating that local ordering did not transfer. Preserve the 0.94989 recent7 pair as the current best. A decision to spend another submission remains with the student; no calibrated score forecast or claim of 0.96+ is justified.

### Explanation and reproduction

For each transaction the score is 0.5 × XGBoost + 0.25 × LightGBM_leaf5 + 0.25 × LightGBM_leaf50. Relative to the current best pair, the adjustment is 0.25 × (LightGBM_leaf50 − LightGBM_leaf5). XGBoost's share stays at 50%. All three use fitting-row weights 0.5 ** ((training_end − time_step) / 7), divided by their mean. Their hyperparameters, feature lists and random seed are unchanged.

The notebook contains all data loading, fitting, checking, final training and CSV generation code. It can fit from the supplied train/test/sample files alone. Previously saved files are not required for fitting. The experiment directory declares the exact recipe and data hashes and supports resuming the same selected run after an interruption; changed data or settings fail the recipe check. Five code cells executed with genuine captured outputs.

Independent verification recalculated every AUC and summary, verified each component mixture and checking ID/label/time-step alignment, and checked full-test CSV indices, row count, probability range and round trip. All three saved models reproduced their final probabilities after reload. Fresh checking and full-test controls matched earlier reference predictions within 1e-7. The final candidate hash is **1292a9cc091327eebcda5217225d311ba22afbafd9bea44985e500cfa9911451**. Original data and previous submissions were preserved.

The new candidate's test-rank correlation with the submitted public best is **0.998775**, with **983 of the top 1,000** shared. These measure similarity, not hidden-label accuracy or a prediction interval.

### Validation and public-score limits

The fixed six-feature XGBoost exclusion was selected using steps 1–32 and then applied to earlier checking cutoffs. This experiment deliberately retains that supplied recipe, so feature selection is not fully contained within each earlier fitting period. Fitting/checking rows themselves are chronological. The final fit uses only preceding supplied training data and does not fit on test rows or answers. The overlapping checks have also been repeatedly used during development.

The previous weighted LightGBM blend scored 0.94787 publicly despite a higher all-15, later-seven and latest-check AUC than both the public-best pair and the new three-model blend. This is direct evidence against treating those local rankings as reliable public-score forecasts. Preserve this negative result in the report. No public AUC is known for the newly generated candidate, and no automatic final-model selection or Kaggle upload occurred.

**AI assistance record:** the student made the fixed mixture choice and retained the existing checking strategy. Claude supplied the original candidate recipe. OpenAI Codex implemented and ran that selected comparison, verified artifacts, recorded the student-confirmed negative submission result and drafted this journal record. The student remains responsible for understanding the code, interpreting results, selecting submissions and accurately disclosing assistance.

Standalone executed code: e1156285_recency_three_model.ipynb. Declared settings, measured checking results, component models, score ledger, final experimental CSV and verification: recency_three_model_results/.

### Every checking result

| Train through | Check steps | Recent7 pair | Weighted LightGBM blend | Fixed three-model blend |
|---:|---|---:|---:|---:|
| 7 | 8–21 | 0.993837 | 0.992928 | 0.993814 |
| 8 | 9–22 | 0.994389 | 0.993828 | 0.994382 |
| 9 | 10–23 | 0.992803 | 0.992652 | 0.992979 |
| 10 | 11–24 | 0.976693 | 0.977536 | 0.976995 |
| 11 | 12–25 | 0.974708 | 0.976843 | 0.976023 |
| 12 | 13–26 | 0.973664 | 0.978060 | 0.977228 |
| 13 | 14–27 | 0.976168 | 0.977212 | 0.977853 |
| 14 | 15–28 | 0.974018 | 0.974775 | 0.974176 |
| 15 | 16–29 | 0.938092 | 0.944504 | 0.940084 |
| 16 | 17–30 | 0.948454 | 0.953096 | 0.950482 |
| 17 | 18–31 | 0.944733 | 0.946929 | 0.942294 |
| 18 | 19–32 | 0.929304 | 0.931286 | 0.928557 |
| 19 | 20–33 | 0.912455 | 0.912977 | 0.910630 |
| 20 | 21–34 | 0.896894 | 0.906131 | 0.901247 |
| 21 | 22–35 | 0.912482 | 0.916627 | 0.913471 |
